
A notebook keeping a set of annotated examples of working with cpl/hdrl at the data/algorithm level, for use
in developing recipes. This is all outside the pipeline framework/abstraction, and is just cpl/hdrl/python. 

I will refer to CPL/HDRL rather than specifying pyCPL/pyHDRL in comments. 

Note: the majority CPL/HDRL methods operate in place, rather than creating a copy. Keep this in mind. 

Some Usful Links: 

https://www.eso.org/sci/software/pycpl/pycpl-site/api/core.html
https://www.eso.org/sci/software/pycpl/pycpl-site/api/drs.html
https://www.eso.org/sci/software/pycpl/pyhdrl-site/api/core.html
https://www.eso.org/sci/software/pycpl/pyhdrl-site/api/func.html


In [ ]:
%matplotlib inline
import cpl
import hdrl
import numpy as np
import matplotlib.pylab as plt
import os

# intput files; change as needed.

dataDir = "/Users/me/METIS_Simulations/output/"

rawName = os.path.join(dataDir,"imgLM","METIS.DARK_2RG_RAW.2027-01-25_00_13_38.fits")
interName = os.path.join(dataDir,"/MASTER_DARK_2RG_2026-03-13T14-53-04-566224.fits")
spotFile = os.path.join(dataDir,"imgLM","METIS.LM_IMAGE_SCI_RAW.2027-01-25_00_01_13.fits")


# Creating and Reading Images

## Reading CPL Data

Reading RAW images into a CPL variable, reading INTERMEDIATE structures into multple CPL variables, 

Note: the read statements are done by number, not name of the extension, which doesn't seem to be an option. 
We probably want to specify by name, and pull the name<->number correspondance from the headers, for general
robustness. 


In [ ]:
# read a RAW file into CPL
# load from extension=1 because primary header has no data

cplImage = cpl.core.Image.load(rawName,extension=1)

# load the property lists from the primary and data headers, respectively

rawProp = cpl.core.PropertyList.load(rawName,0)
rawDataProp = cpl.core.PropertyList.load(rawName,1)

# read an intermediate, 3 extension FITS file into three CPL variables
# note that I specify the type for the mask frame, to ensure the result
# is an integer

dataFrame = cpl.core.Image.load(interName,extension=1)
noiseFrame = cpl.core.Image.load(interName,extension=2)
maskFrame = cpl.core.Image.load(interName,dtype=cpl.core.Type.INT,extension=3)

# and the property lists

interProp = cpl.core.PropertyList.load(interName,0)
dataProp = cpl.core.PropertyList.load(interName,1)
noiseProp = cpl.core.PropertyList.load(interName,2)
maskProp = cpl.core.PropertyList.load(interName,3)



## Bit Masks

CPL image consist of data plus a 1 bit True-False map for masked/bad pixels. HDRL images contain two CPL images, one for data and one for errors/noise.  The bit mask is read by CPL/HDRL routines, much like a python masked array. 

METIS FITS data products contain three extensions; data, errors and a 32 bit mask encoding different types of flagged pixels. We plan to represent this as an HDRL image (data and noise) and a CPL image (32 bit mask). Relevant flags will be copied from the CPL image to the data bitmask before being passed to CPL/HDRL routines, or from the HDRL/CPL image to the 32 bit mask after processing. 

The following two routines do that copying. 





In [ ]:
def applyMask(cplImage,cplMask,bits):

    """
    apply a 32 bit mask contained in a cpl image to another cplImage.
    
    intput: 
       cplImage: image
       cplMask: 32 bit integer mask
       bits: list of bits apply: [1,2,4,8...]

    output:
       new cpl image with applied mask
    """
    
    # create a new CPL image to hold the mask
    maskFrame = cpl.core.Image.zeros_like(cplMask)
    
    # for each bit that should be selected, do a bitwise and to extract
    # the value, then add to the mask above
    for bit in bits:
        temp = cpl.core.Image.zeros_like(maskFrame)
        temp.copy_into(cplMask,0,0)
        temp.and_scalar(bit)
        maskFrame.add(temp)

    # create a mask object
    mask = cpl.core.Mask(cplImage.shape[0],cplImage.shape[1])

    # a bit kludgy, but creating a numpy boolean array with the 
    # required mask values, then directly assigning it in the way
    # pycpl accepts indices. TODO get pycpl native way of doing this. 
    
    isTrue = maskFrame.as_array().astype(bool)
    mask[0:cplImage.shape[0]][0:cplImage.shape[1]] = isTrue

    #apply the mask to the cplImage. This overwrites the mask
    cplImage.reject_from_mask(mask)
    return cplImage

def updateMask(image, cplMask, bitVal):

    """
    Take the 1 bit mask from an HDRL or CPL image and use it to update a
    32 bit METIS bad pixel mask
    
    Input: 
       image: HDRL or CPL image
       cplMask: CPL image of type INT containing METIS BPM
       bitVal: bit value for new flag. Must by power of 2. 
    """

    # extract the mask into numpy array. 
    # input can be HDRL image or CPL image
    try: 
        mask = image.image.bpm.as_array()
    except:
        mask = image.bpm.as_array()

    # turn it into a CPL image
    update = cpl.core.Image(mask,dtype=cpl.core.Type.INT)

    # multiply it by the bit value

    update.multiply_scalar(bitVal)

    #and update the mask
    cplMask.add(update)

    return cplMask


In [ ]:
# use the function defined below to apply the mask. 

newImage = applyMask(dataFrame,maskFrame,[1])


## Creating an HDRL image

Creating an HDRL image from the above.  The HDRL image contains two CPL images, one with data
and one with errors, as well as a 1 bit mask. We will store the data and noise extensions
correspondingly, and have a routine that takes the 32 bit mask from the FITS files, and creates
a 1 bit mask by filtering which bits we need to copy over (see routine applyMask)


In [ ]:
hdrlImage = hdrl.core.Image(dataFrame,noiseFrame)


## HDRL ImageList

Here I'm taking the first image, copying it, multiplying it by a scalar, and adding it to the list. The multiple scalar method takes to scalars; one for the data, one for the noise. The collapse option returns an image and a contribution mask. 

In [ ]:


#create an image, then copy into it, then multiply in place. 
a = hdrl.core.Image.zeros(cplImage.shape[0],cplImage.shape[1])
a.copy_into(hdrlImage,0,0)
a.mul_scalar([3,3])

b = hdrl.core.Image.zeros(cplImage.shape[0],cplImage.shape[1])
b.copy_into(hdrlImage,0,0)
b.mul_scalar([2,2])

# create an empty HDRL image list and append.
hdrlList = hdrl.core.ImageList()

hdrlList.append(hdrlImage)
hdrlList.append(a)
hdrlList.append(b)

out1, out2 = hdrlList.collapse_median()


# Image Manipulation

## CPL Operations

In [ ]:

# read in two images
image1 = cpl.core.Image.load(rawName,extension=1)
image2 = cpl.core.Image.load(rawName,extension=1)

# a statistical function

print(f"CPL Mean: {image1.get_mean()}")

# we can access the underlying numpy array (minus bitmask)

print(f"Numpy Mean: {image1.as_array().mean()}")

# but if we do the same two things on an array with a bitmask, we get different values

print(f"CPL Mean: {newImage.get_mean()}")
print(f"Numpy Mean: {newImage.as_array().mean()}")


# multiply image1 by image2

image1.multiply(image2)

# multiple image1 by a scalar

image1.multiply_scalar(0.5)

# get some statistics

# create a gaussian kernel 101x101 pixels, centred, with sigma of 40x30
kernel = cpl.core.Image.create_gaussian(501,501,cpl.core.Type.DOUBLE,251,251,1000,200,300)
kernel.multiply_scalar(10000)

# copy the Gaussian into image2, near the centre

image1.copy_into(kernel,1000,1000)


In [ ]:
# note that we can pass the CPL image to matplotlib as we would a numpy array

fig,ax=plt.subplots()
ax.imshow(image1)

## Point source detection

Using the HDRL catalogue function
Code snippets copied from the HDRL API docs

The returned catalogue is a cpl.core.Table 


In [ ]:
# decreasing sequence of thresholds
thresh = cpl.core.Vector([3,2,1])

#load an example image, and the WCS
image = cpl.core.Image.load(spotFile,extension=1)
wcs = cpl.drs.WCS(cpl.core.PropertyList.load(spotFile,1))

# Create catalogue object, with input parameters
cat = hdrl.func.Catalogue(
    obj_min_pixels=10,
    obj_threshold=3.0,
    obj_deblending=True,
    obj_core_radius=2.0,
    bkg_estimate=True,
    bkg_mesh_size=64,
    bkg_smooth_fwhm=2.0,
    det_eff_gain=1.0,
    det_saturation=65535.0,
    resulttype=7
)

# Compute catalogue
result = cat.compute(image, wcs=wcs) # hdrl funct catalogue result

# Access results
catalogue = result.catalogue  # cpl Table
segmap = result.segmentation_map # cpl Image
background = result.background # cpl Image
qc_info = result.qclist # cpl PropertyList


## CPL tables

and the catalouge results. 

In [ ]:
# look at the column names
for elem in catalogue.column_names:
    print(elem)


In [ ]:
for i in range(len(catalogue)):
    print(catalogue['X_coordinate'].as_array[i],catalogue['Y_coordinate'].as_array[i],catalogue['Classification'].as_array[i])


In [ ]:
# sort by a column (in place)

# some weirdness here; the following works, but switching to cpl.core.Sort.DESCENDING 
# gives a warning about typecasting INT to BOOL, and changing the type to INT gives a 
# type mismatch.  Maybe expecting a boolean, while ASCENDING/DESCENDING have values of 1, -1.

# also, sort by ASCENDING looks more like DESCENDING when you print out the results. 

# pass the sort method a propertylist. Each property gives the name of the column, the stype of cpl.core.Sort (see above)
# and cpl.core.Sort.ASCENDING / cpl.core.Sort.DESCENDING; the columns are sorted in sequence. 

sortBy = cpl.core.PropertyList()

# giving it Classifcation because that has identical values in some rows, then X position. 

sortBy.append(cpl.core.Property("Classification",cpl.core.Type.BOOL,cpl.core.Sort.ASCENDING))
sortBy.append(cpl.core.Property("X_coordinate",cpl.core.Type.BOOL,cpl.core.Sort.ASCENDING))

catalogue.sort(sortBy)

In [ ]:
# check the results
for i in range(len(catalogue)):
    print(catalogue['X_coordinate'].as_array[i],catalogue['Y_coordinate'].as_array[i],catalogue['Classification'].as_array[i])


In [ ]:
#look at a specific element
#this returns a tuple; the value and the mask

print(catalogue['X_coordinate'][0])


In [ ]:
# here we do need to pass the as_array, otherwise scatter gets a list of tuples and chokes

fig,ax=plt.subplots()
ax.scatter(catalogue['X_coordinate'].as_array,catalogue['Y_coordinate'].as_array,c=catalogue['Isophotal_flux'].as_array)

## List reading functions

To read a list of files into an CPL imageList (RAW) or HDRL imagelist (Intermediate). The latter
applies the bitmask extention to the HDRL image. 

In [ ]:
def readRawList(fileNames):

    """ read a list of RAW files into a CPL image list
        returns the first property list as well. """

    cplList = cpl.core.ImageList()

    propList = cpl.core.PropertyList.load(fileNames[0],extension=1)
    
    for fName in fileNames: 
        cplList.append(cpl.core.Image.load(rawName,extension=1))

    return cplList,propList

def readInterList(fileNames,bits):

    """ read a list of 3 extension files into a hdrl image list 
        returns the first property list as well. """
    
    hdrlList = hdrl.core.ImageList()
    propList = cpl.core.PropertyList.load(fileNames[0],extension=0)

    for fName in fileNames:
        data = cpl.core.Image.load(rawName,extension=1)
        noise = cpl.core.Image.load(rawName,extension=2)
        mask = cpl.core.Image.load(rawName,extension=3)

        data = applyMask(data,mask,bits)

        hdrlList.append(hdrl.core.Image(data,noise))
    return hdrlList, propList
    

# Vectors

In [ ]:

# quick test of the applyMask routine, to a 2x2 image

dataFrame = cpl.core.Image.zeros(2,2,cpl.core.Type.FLOAT)
dataFrame[0][0] = 1
dataFrame[0][1] = 1
maskFrame=cpl.core.Image.zeros(2,2,cpl.core.Type.INT)
maskFrame[0][1] = 1
maskFrame[0][0] = 2

#print(dataFrame)

newImage = applyMask(dataFrame,maskFrame,[1])
newImage.bpm[0][0],newImage.bpm[0][1], newImage.bpm[1][0],newImage.bpm[1][1]
